In [1]:
from __future__ import annotations

import os
import sys
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output

# ML
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
def find_site_root(start: Path) -> Path:
    """
    Find site root by walking upwards until we find:
      - etc/config.json
      - src/
    """
    cur = start.resolve()
    for _ in range(8):
        if (cur / "etc" / "config.json").exists() and (cur / "src").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find site root containing etc/config.json and src/")

SITE_ROOT = find_site_root(Path.cwd())
SRC = SITE_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("SITE_ROOT =", SITE_ROOT)
print("SRC      =", SRC)

SITE_ROOT = /home/johnathanc/APM/LIO
SRC      = /home/johnathanc/APM/LIO/src


In [3]:
from apm_core.ssot import load_ssot, list_sensors, get_sensor
from apm_core.settings import load_ini
from apm_core.db_interface import DBInterface
from apm_core.raw_pull import build_wide_frame

import logging

ssot = load_ssot(SITE_ROOT)
global_debug = bool(ssot.get("debug", False))

settings = load_ini(SITE_ROOT, debug=global_debug)

nb_logger = logging.getLogger("TRAINING")
nb_logger.setLevel(logging.INFO)
if not nb_logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    nb_logger.addHandler(h)

db = DBInterface(settings=settings, logger=nb_logger)

SENSORS = list_sensors(ssot)

print("Loaded sensors:", len(SENSORS))
print(SENSORS)

2026-02-25 14:12:09,814 | INFO | PostgreSQL pool initialised
2026-02-25 14:12:09,833 | INFO | MSSQL connected | table=APM_Scalability_Dev


Loaded sensors: 4
['Mill1PinionConditionScore', 'Mill1TrunnionConditionScore', 'Mill1MotorConditionScore', 'Mill1GearboxConditionScore']


In [4]:
sensor_dropdown = widgets.Dropdown(
    options=["ALL"] + SENSORS,
    value="ALL",
    description="Sensor:",
    layout=widgets.Layout(width="450px")
)

start_picker = widgets.Text(
    value=(datetime.utcnow().replace(microsecond=0) - pd.Timedelta(days=14)).strftime("%Y-%m-%d %H:%M:%S"),
    description="Start UTC:",
    layout=widgets.Layout(width="450px")
)

end_picker = widgets.Text(
    value=datetime.utcnow().replace(microsecond=0).strftime("%Y-%m-%d %H:%M:%S"),
    description="End UTC:",
    layout=widgets.Layout(width="450px")
)

btn_train = widgets.Button(description="Train Isolation Forest", button_style="success", icon="check")
out = widgets.Output()

ui = widgets.VBox([
    sensor_dropdown,
    widgets.HBox([start_picker, end_picker]),
    btn_train,
    out
])

display(ui)

In [5]:
def get_iforest_training_cfg(sensor_cfg: Dict[str, Any]) -> Dict[str, Any]:
    method = (sensor_cfg.get("Method", {}) or {}).get("IsolationForest", {}) or {}
    training = method.get("Training", {}) or {}

    # SSOT-only defaults if missing
    return {
        "pull_interval": str(training.get("pull_interval", "15 minutes")),
        "resample_rule": str(training.get("resample_rule", "h")),
        "dropna": str(training.get("dropna", "any")),
        "n_estimators": int(training.get("n_estimators", 200)),
        "contamination": float(training.get("contamination", 0.01)),
        "max_samples": training.get("max_samples", "auto"),
        "random_state": int(training.get("random_state", 42)),
    }

In [6]:
def parse_utc(dt_str: str) -> datetime:
    # expects "YYYY-mm-dd HH:MM:SS"
    return datetime.strptime(dt_str.strip(), "%Y-%m-%d %H:%M:%S")

In [7]:
def compute_gate_open_mask(
    *,
    index: pd.DatetimeIndex,
    filter_series: pd.Series,
    filter_value: float,
    exclude_startup: bool,
    startup_minutes: int,
) -> pd.Series:
    """
    Gate open where filter_series >= filter_value.
    Optionally remove a startup window after each closed->open transition.
    """
    s = filter_series.reindex(index).ffill()
    open_mask = (s >= float(filter_value)).fillna(False)

    if not exclude_startup or startup_minutes <= 0:
        return open_mask

    prev_open = open_mask.shift(1).fillna(False)
    transitions = (~prev_open) & (open_mask)  # closed -> open

    mask = open_mask.copy()
    for t in index[transitions]:
        end = t + pd.Timedelta(minutes=int(startup_minutes))
        mask.loc[(index >= t) & (index <= end)] = False

    return mask

In [8]:
def pull_training_frame(
    *,
    sensor_name: str,
    start: datetime,
    end: datetime,
) -> Tuple[pd.DataFrame, Dict[str, Any], Dict[str, Any]]:
    sensor = get_sensor(ssot, sensor_name)
    cfg = sensor.cfg
    train_cfg = get_iforest_training_cfg(cfg)

    resample_rule = train_cfg["resample_rule"]
    dropna = train_cfg["dropna"]
    pull_interval = pd.Timedelta(train_cfg["pull_interval"])

    # map displayname -> raw tag
    feature_map: Dict[str, str] = {}
    for disp, meta in (cfg.get("Features", {}) or {}).items():
        if isinstance(meta, dict) and meta.get("tag"):
            feature_map[disp] = meta["tag"]

    # gate config
    ft = cfg.get("filter_tag", {}) or {}
    other = cfg.get("Other", {}) or {}
    filter_value = float(ft.get("filter_value", other.get("filter_value", 0.9)))
    startup_minutes = int(other.get("startup_period", 0))

    # chunked pull to avoid massive queries
    X_parts = []
    t0 = pd.Timestamp(start)
    t1 = pd.Timestamp(end)

    chunk = pd.Timedelta(days=2)  # ✅ safe default; change to 1 day if still heavy
    cur = t0

    while cur < t1:
        nxt = min(cur + chunk, t1)

        df_wide, filter_tag_series = build_wide_frame(db=db, sensor=sensor, start=cur.to_pydatetime(), end=nxt.to_pydatetime())
        if df_wide is None or df_wide.empty:
            cur = nxt
            continue

        # UTC-naive
        df_wide = df_wide.copy()
        df_wide.index = pd.to_datetime(df_wide.index, utc=True).tz_convert(None)
        df_wide = df_wide.sort_index()

        if filter_tag_series is None or filter_tag_series.empty:
            filter_tag_series = pd.Series(index=df_wide.index, data=np.nan, dtype=float)
        else:
            filter_tag_series = filter_tag_series.copy()
            filter_tag_series.index = pd.to_datetime(filter_tag_series.index, utc=True).tz_convert(None)
            filter_tag_series = filter_tag_series.sort_index()

        # build feature matrix
        X = pd.DataFrame(index=df_wide.index)
        for disp, tag in feature_map.items():
            X[disp] = pd.to_numeric(df_wide.get(tag, np.nan), errors="coerce")

        # Gate filter + exclude startup (SSOT-only)
        gate_mask = compute_gate_open_mask(
            index=X.index,
            filter_series=filter_tag_series,
            filter_value=filter_value,
            exclude_startup=True,
            startup_minutes=startup_minutes,
        )
        X = X.loc[gate_mask].copy()

        # ✅ downsample EARLY using pull_interval (config.json)
        if pull_interval is not None and pull_interval.total_seconds() > 0:
            X = X.resample(pull_interval).mean()

        X_parts.append(X)
        cur = nxt

    if not X_parts:
        raise ValueError(f"{sensor_name}: no usable data after chunked pulls")

    X_all = pd.concat(X_parts).sort_index()

    # final resample (hourly etc)
    if resample_rule and str(resample_rule).strip():
        X_all = X_all.resample(str(resample_rule)).mean()

    # dropna strategy
    dropna = str(dropna).lower().strip()
    if dropna == "any":
        X_all = X_all.dropna(how="any")
    elif dropna == "all":
        X_all = X_all.dropna(how="all")
    elif dropna == "none":
        pass
    else:
        X_all = X_all.dropna(how="any")

    meta = {
        "sensor": sensor_name,
        "start_utc": start.strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc": end.strftime("%Y-%m-%d %H:%M:%S"),
        "pull_interval": str(train_cfg["pull_interval"]),
        "resample_rule": resample_rule,
        "dropna": dropna,
        "filter_value": filter_value,
        "exclude_startup": True,
        "startup_minutes": startup_minutes,
        "n_rows_after_filters": int(len(X_all)),
        "n_features": int(X_all.shape[1]),
        "features": list(X_all.columns),
    }
    return X_all, meta, train_cfg

In [9]:
from pathlib import Path

def get_output_dir(sensor_name: str) -> Path:
    return SITE_ROOT / "etc" / "trained_params" / sensor_name / "IsolationForest"

In [10]:
def train_and_save_iforest(
    *,
    sensor_name: str,
    start: datetime,
    end: datetime,
) -> Dict[str, Any]:
    print(f"[{sensor_name}] Pulling + preprocessing training frame...")
    X, meta, train_cfg = pull_training_frame(sensor_name=sensor_name, start=start, end=end)
    print(f"[{sensor_name}] After filters: rows={len(X)} cols={X.shape[1]} resample={meta['resample_rule']} dropna={meta['dropna']}")

    if X.empty or len(X) < 50:
        raise ValueError(f"{sensor_name}: not enough training rows after filtering (rows={len(X)}).")

    print(f"[{sensor_name}] Scaling...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)

    ms = train_cfg["max_samples"]
    if isinstance(ms, str) and ms.strip().lower() != "auto":
        try:
            ms = int(ms)
        except Exception:
            ms = "auto"

    print(f"[{sensor_name}] Fitting IsolationForest (n_estimators={train_cfg['n_estimators']}, contamination={train_cfg['contamination']}, max_samples={train_cfg['max_samples']}) ...")
    model = IsolationForest(
        n_estimators=int(train_cfg["n_estimators"]),
        contamination=float(train_cfg["contamination"]),
        max_samples=ms,
        random_state=int(train_cfg["random_state"]),
        n_jobs=-1,
    )
    model.fit(X_scaled)

    print(f"[{sensor_name}] Computing diagnostics...")
    scores = model.score_samples(X_scaled)
    pred = model.predict(X_scaled)
    frac_anom = float((pred == -1).mean())

    outdir = get_output_dir(sensor_name)
    outdir.mkdir(parents=True, exist_ok=True)  # ✅ ensures folder exists
    print(f"[{sensor_name}] Saving to: {outdir}")

    model_path = outdir / "trained_isolation_forest.joblib"
    meta_path = outdir / "trained_isolation_forest.json"

    payload = {
        "ok": True,
        "sensor": sensor_name,
        "method": "IsolationForest",
        "trained_at_utc": datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
        "window": {"start_utc": meta["start_utc"], "end_utc": meta["end_utc"]},
        "preprocess": {
            "resample_rule": meta["resample_rule"],
            "dropna": meta["dropna"],
            "exclude_startup": meta["exclude_startup"],
            "startup_minutes": meta["startup_minutes"],
            "filter_value": meta["filter_value"],
        },
        "features": meta["features"],
        "shape": {"rows": meta["n_rows_after_filters"], "cols": meta["n_features"]},
        "params": {
            "n_estimators": int(train_cfg["n_estimators"]),
            "contamination": float(train_cfg["contamination"]),
            "max_samples": str(train_cfg["max_samples"]),
            "random_state": int(train_cfg["random_state"]),
            "pull_interval": train_cfg["pull_interval"],
        },
        "diagnostics": {
            "frac_predicted_anomaly": frac_anom,
            "score_samples_min": float(np.min(scores)),
            "score_samples_p05": float(np.quantile(scores, 0.05)),
            "score_samples_median": float(np.median(scores)),
            "score_samples_p95": float(np.quantile(scores, 0.95)),
            "score_samples_max": float(np.max(scores)),
        },
        "artifacts": {"model_joblib": str(model_path), "meta_json": str(meta_path)},
    }

    bundle = {"model": model, "scaler": scaler, "feature_columns": meta["features"], "meta": payload}
    joblib.dump(bundle, model_path)

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"[{sensor_name}] ✅ Saved:\n  - {model_path}\n  - {meta_path}")
    return payload

In [11]:
def train_selected(_):
    with out:
        clear_output()
        try:
            sensor_sel = sensor_dropdown.value

            start = parse_utc(start_picker.value)
            end = parse_utc(end_picker.value)
            if end <= start:
                raise ValueError("End must be after Start.")

            sensors_to_train = SENSORS if sensor_sel == "ALL" else [sensor_sel]

            print("Training Isolation Forest for:", sensors_to_train)
            print("Start UTC:", start, "| End UTC:", end)
            print()

            for sname in sensors_to_train:
                print("=" * 80)
                print("SENSOR:", sname)
                payload = train_and_save_iforest(sensor_name=sname, start=start, end=end)
                print("Saved:", payload["artifacts"]["model_joblib"])
                print("Saved:", payload["artifacts"]["meta_json"])
                print("Diagnostics:", payload["diagnostics"])

            print("\n✅ DONE")

        except Exception as e:
            print("❌ ERROR:", repr(e))

btn_train.on_click(train_selected)

In [12]:
def list_outputs(sensor_name: str):
    outdir = get_output_dir(sensor_name)
    if not outdir.exists():
        print("No output directory:", outdir)
        return
    print("Output dir:", outdir)
    for p in sorted(outdir.glob("*")):
        print(" -", p.name)

# example
list_outputs(sensor_dropdown.value if sensor_dropdown.value != "ALL" else SENSORS[0])

Output dir: /home/johnathanc/APM/LIO/etc/trained_params/Mill1PinionConditionScore/IsolationForest
 - .ipynb_checkpoints
 - trained_isolation_forest.joblib
 - trained_isolation_forest.json
